# 13 — Error Handling and Retries

Every tool accepts retry-related parameters (see [`docs/05_tools_reference.md`](../docs/05_tools_reference.md)): maximum retries, retry delay, retry delay maximum, retry mode (`fixed`/`exponential`), and a network timeout. This notebook shows how to apply the same ideas on the *client* side too.

In [ ]:
import asyncio
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.mcp_client import connect

async def call_with_backoff(namespace, tool, args=None, max_retries=3, base_delay=1.0):
    last_exc = None
    for attempt in range(max_retries):
        try:
            async with connect(namespaces=[namespace], read_only=True) as client:
                return await client.call_tool(tool, args or {})
        except Exception as exc:
            last_exc = exc
            delay = base_delay * (2 ** attempt)
            print(f"Attempt {attempt + 1} failed ({exc}); retrying in {delay:.1f}s")
            await asyncio.sleep(delay)
    raise last_exc

In [ ]:
# Bound how long we're willing to wait for a single call, independent of
# how many retries remain.
async def call_with_timeout(namespace, tool, args=None, timeout_seconds=15):
    async with connect(namespaces=[namespace], read_only=True) as client:
        return await asyncio.wait_for(client.call_tool(tool, args or {}), timeout=timeout_seconds)

try:
    result = await call_with_timeout("subscription", "azmcp_subscription_list")
    print(result.content)
except asyncio.TimeoutError:
    print("Call timed out - the server may be slow to start on first run.")

In [ ]:
# Common failure categories worth distinguishing in your own error handling:
# 1) Connection/startup failures (Node.js missing, npx download issues)
# 2) Authentication failures (not signed in, wrong tenant)
# 3) Authorization/RBAC failures (signed in, but lacking the role)
# 4) Application-level errors returned inside a successful MCP response
async def classify_and_call(namespace, tool, args=None):
    try:
        async with connect(namespaces=[namespace], read_only=True) as client:
            result = await client.call_tool(tool, args or {})
            return result
    except FileNotFoundError:
        print("Startup failure: is Node.js installed?")
    except Exception as exc:
        print(f"Runtime failure: {exc}")

await classify_and_call("group", "azmcp_group_list")

## Logging best practices

```python
import logging
logging.basicConfig(level=logging.INFO)
```

Log the tool **name** and **non-secret** arguments for every call, but never log `.content` from tools annotated `Secret` (see notebook 14) — route those through elicitation instead.